# Phase 1 — Exploratory Data Analysis (EDA)
## Air Quality Index Analysis — Indian Cities

**Dataset:** `data/raw/city_day.csv`  
**Source:** [Kaggle — Air Quality Data in India](https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india)  
**Output figures:** `reports/figures/`

---

## Section 1 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

# Output directory
os.makedirs('reports/figures', exist_ok=True)

# Load dataset
df = pd.read_csv('data/raw/city_day.csv', parse_dates=['Date'])
print("Shape:", df.shape)
df.head()

---
## Section 2 — Basic Dataset Overview

In [ ]:
# Data types and non-null counts
df.info()

# Summary statistics for numeric columns
df.describe().T

In [ ]:
# Total missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print(missing_df[missing_df['Missing Count'] > 0])

# Unique cities in dataset
print("\nCities:", df['City'].unique())
print("Date range:", df['Date'].min(), "to", df['Date'].max())
print("Total rows:", len(df))

---
## Section 3 — AQI Distribution (Overall)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of AQI values
axes[0].hist(df['AQI'].dropna(), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Overall AQI Distribution')
axes[0].set_xlabel('AQI Value')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['AQI'].mean(), color='red', linestyle='--', label=f"Mean: {df['AQI'].mean():.1f}")
axes[0].legend()

# AQI_Bucket class distribution (bar chart)
bucket_order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
bucket_counts = df['AQI_Bucket'].value_counts().reindex(bucket_order)
colors = ['#2ecc71', '#a8d08d', '#f9c74f', '#f77f00', '#d62828', '#6a040f']

axes[1].bar(bucket_counts.index, bucket_counts.values, color=colors, edgecolor='white')
axes[1].set_title('AQI Bucket Class Distribution')
axes[1].set_xlabel('AQI Bucket')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('reports/figures/01_aqi_distribution.png', bbox_inches='tight')
plt.show()

# Print class imbalance stats
print("\nClass Distribution (%):")
print((df['AQI_Bucket'].value_counts(normalize=True) * 100).round(2))

---
## Section 4 — City-wise AQI Analysis

In [ ]:
target_cities = ['Delhi', 'Bengaluru', 'Kolkata', 'Hyderabad']
city_df = df[df['City'].isin(target_cities)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot — AQI by city
sns.boxplot(data=city_df, x='City', y='AQI', palette='Set2', ax=axes[0])
axes[0].set_title('AQI Distribution by City')
axes[0].set_xlabel('City')
axes[0].set_ylabel('AQI Value')

# Mean AQI bar chart per city
city_mean = city_df.groupby('City')['AQI'].mean().sort_values(ascending=False)
axes[1].bar(city_mean.index, city_mean.values, color=sns.color_palette('Set2', 4))
axes[1].set_title('Mean AQI by City')
axes[1].set_xlabel('City')
axes[1].set_ylabel('Mean AQI')

for i, v in enumerate(city_mean.values):
    axes[1].text(i, v + 2, f'{v:.1f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('reports/figures/02_city_aqi.png', bbox_inches='tight')
plt.show()

---
## Section 5 — Time-Series AQI Trends

In [ ]:
# Monthly average AQI per city
city_df = city_df.copy()
city_df['YearMonth'] = city_df['Date'].dt.to_period('M').dt.to_timestamp()

monthly_avg = city_df.groupby(['YearMonth', 'City'])['AQI'].mean().reset_index()

plt.figure(figsize=(16, 5))
for city in target_cities:
    subset = monthly_avg[monthly_avg['City'] == city]
    plt.plot(subset['YearMonth'], subset['AQI'], label=city, linewidth=1.8)

plt.title('Monthly Average AQI Trend by City')
plt.xlabel('Date')
plt.ylabel('Average AQI')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('reports/figures/03_monthly_aqi_trend.png', bbox_inches='tight')
plt.show()

In [ ]:
# Season-wise AQI
city_df['Month'] = city_df['Date'].dt.month
city_df['Season'] = city_df['Month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Monsoon', 10: 'Monsoon', 11: 'Monsoon'
})

plt.figure(figsize=(12, 5))
season_order = ['Winter', 'Spring', 'Summer', 'Monsoon']
sns.boxplot(data=city_df, x='Season', y='AQI',
            order=season_order, hue='City', palette='Set2')
plt.title('Season-wise AQI by City')
plt.xlabel('Season')
plt.ylabel('AQI')
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('reports/figures/04_season_aqi.png', bbox_inches='tight')
plt.show()

---
## Section 6 — Pollutant Correlation Heatmap

In [ ]:
pollutants = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx',
              'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI']

corr_df = df[pollutants].dropna()
corr_matrix = corr_df.corr()

plt.figure(figsize=(13, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title('Pollutant Correlation Heatmap')
plt.tight_layout()
plt.savefig('reports/figures/05_correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Top correlations with AQI
print("\nTop correlations with AQI:")
print(corr_matrix['AQI'].drop('AQI').sort_values(ascending=False))

---
## Section 7 — Pollutant Distributions (Delhi Only)

In [ ]:
delhi_df = df[df['City'] == 'Delhi'].copy()

pollutant_cols = ['PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(pollutant_cols):
    axes[i].hist(delhi_df[col].dropna(), bins=40,
                 color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'{col} Distribution (Delhi)')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(delhi_df[col].mean(), color='red',
                    linestyle='--', linewidth=1.2,
                    label=f"Mean: {delhi_df[col].mean():.2f}")
    axes[i].legend(fontsize=8)

plt.suptitle('Key Pollutant Distributions — Delhi', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('reports/figures/06_delhi_pollutant_dist.png', bbox_inches='tight')
plt.show()

---
## Section 8 — Key Observations

### Class Imbalance
- AQI_Bucket is heavily skewed toward **'Moderate'** and **'Poor'**
- **'Severe'** and **'Good'** are rare classes — SMOTE required in Phase 2

### City Insights
- **Delhi** consistently has the highest AQI across all seasons
- **Bengaluru** has the lowest mean AQI among the four cities
- Winter months drive AQI spikes in Delhi and Kolkata

### Pollutant Correlations
- **PM2.5** and **PM10** have the strongest correlation with AQI
- **CO** and **NO2** are moderately correlated with AQI
- **O3** shows a weaker or inverse correlation in some cities

### Missing Data
- Several pollutant columns have significant missing values
- Imputation strategy will be defined in **Phase 2 (Preprocessing)**